# Colab: ImageNet SSL 预训练（ViT-Tiny）

这个 notebook 按当前仓库的 `train_ssl_freq_mae.py` 写好了 Colab 流程，目标是：

1. 克隆 `https://github.com/BoldHu/test.git`
2. 准备 ImageNet 数据，并放到仓库能直接识别的位置
3. 先做一个小型 smoke test
4. smoke test 通过后，再正式启动 `train_ssl_freq_mae.py` 的 ViT-Tiny 预训练

默认策略：

- 数据尽量放在 `/content`，这是 Colab 里最快的读取路径
- 日志和 checkpoint 放到 Google Drive，避免断连丢失
- 如果你担心 230G 本地磁盘不够，可以把 `USE_DRIVE_FOR_DATA` 改成 `True`
- Kaggle 路径优先兼容 `ILSVRC/Data/CLS-LOC/{train,val}`，也兼容 `train/val`


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/BoldHu/test.git'
REPO_DIR = Path('/content/test')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'timm>=1.0.0',
        'einops>=0.7.0',
        'tensorboard>=2.16.0',
        'matplotlib>=3.7.0',
        'tqdm>=4.65.0',
        'datasets>=2.20.0',
        'huggingface_hub>=0.24.0',
        'pillow>=10.0.0',
    ],
    check=True,
)

print('Repo ready:', REPO_DIR)


In [ ]:
from pathlib import Path
import os
import shutil
import torch

HF_DATASET_ID = 'ILSVRC/imagenet-1k'
HF_ACCESS_URL = 'https://huggingface.co/datasets/ILSVRC/imagenet-1k'

# 数据和 HF 缓存都放到 Colab 本地盘，训练更快，也避免 Drive 小文件写入过慢。
DRIVE_ROOT = Path('/content/drive/MyDrive')
HF_HOME = Path('/content/.cache/huggingface')
HF_CACHE_ROOT = HF_HOME / 'datasets'
DATA_ROOT = Path('/content/imagenet_hf')
RUNS_ROOT = DRIVE_ROOT / 'imagenet_ssl_runs'
SMOKE_ROOT = Path('/content/imagenet_smoke')
MAX_CONTENT_USED_GB = 200.0
MIN_CONTENT_FREE_GB = 25.0
DELETE_HF_CACHE_AFTER_EXPORT = True

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE_ROOT)

HF_HOME.mkdir(parents=True, exist_ok=True)
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

# ?? ViT backbone??????????????
TIMM_TINY_MODEL = 'vit_tiny_patch16_224.augreg_in21k_ft_in1k'
FULL_EXPERIMENT = 'vit_tiny_ssl_imagenet_colab'
SMOKE_EXPERIMENT = 'vit_tiny_ssl_smoke'
FULL_EPOCHS = 800

if not torch.cuda.is_available():
    raise RuntimeError('?? Colab ??? GPU Runtime ?????')

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

if gpu_mem_gb >= 35:
    TRAIN_BATCH_SIZE = 128
    VAL_BATCH_SIZE = 128
    GRAD_ACCUM_STEPS = 2
    NUM_WORKERS = 4
elif gpu_mem_gb >= 20:
    TRAIN_BATCH_SIZE = 64
    VAL_BATCH_SIZE = 64
    GRAD_ACCUM_STEPS = 4
    NUM_WORKERS = 4
else:
    TRAIN_BATCH_SIZE = 32
    VAL_BATCH_SIZE = 32
    GRAD_ACCUM_STEPS = 8
    NUM_WORKERS = 2

effective_bs = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS

def print_disk(path: Path):
    usage = shutil.disk_usage(path)
    print(f'{path}: free={usage.free / 1024**3:.1f} GiB | used={usage.used / 1024**3:.1f} GiB | total={usage.total / 1024**3:.1f} GiB')

print('GPU:', gpu_name)
print(f'GPU memory: {gpu_mem_gb:.1f} GiB')
print('HF dataset:', HF_DATASET_ID)
print('HF access page:', HF_ACCESS_URL)
print('HF cache root:', HF_CACHE_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('RUNS_ROOT:', RUNS_ROOT)
print('TIMM_TINY_MODEL:', TIMM_TINY_MODEL)
print('Suggested batch size:', TRAIN_BATCH_SIZE)
print('Suggested grad_accum_steps:', GRAD_ACCUM_STEPS)
print('Effective batch size:', effective_bs)
print('MAX_CONTENT_USED_GB:', MAX_CONTENT_USED_GB)
print('MIN_CONTENT_FREE_GB:', MIN_CONTENT_FREE_GB)
print_disk(Path('/content'))
print_disk(DRIVE_ROOT)


## Hugging Face ??

???????? Kaggle?

????? Hugging Face ?????????

- ?? `ILSVRC/imagenet-1k` ??? `Access repository`?`https://huggingface.co/datasets/ILSVRC/imagenet-1k`
- ? Hugging Face Settings ????? `read` token

???????? token???????? gated dataset ??????


In [ ]:
from getpass import getpass
import os

from huggingface_hub import HfApi, login

HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if not HF_TOKEN:
    HF_TOKEN = getpass('Paste your Hugging Face read token (starts with hf_): ').strip()

if not HF_TOKEN or not HF_TOKEN.startswith('hf_'):
    raise ValueError('??? Hugging Face token?read token ??? hf_ ???')

os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

api = HfApi(token=HF_TOKEN)
try:
    info = api.dataset_info(HF_DATASET_ID, token=HF_TOKEN)
except Exception as exc:
    raise RuntimeError(
        'Hugging Face ?????????'
        '\n1. ????????? Access repository'
        '\n2. token ???????? read token'
        f'\n3. ???????: {HF_DATASET_ID}'
    ) from exc

print('HF login is ready.')
print('Dataset id:', info.id)
print('Private/gated access confirmed.')


In [ ]:
from pathlib import Path
import itertools
import json
import os
import shutil

from datasets import load_dataset
from PIL import Image

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.JPG', '.JPEG', '.PNG'}
HF_SPLIT_TO_DIR = {'train': 'train', 'validation': 'val'}

def find_images(root: Path, limit: int | None = None):
    images = []
    for path in sorted(root.rglob('*')):
        if path.is_file() and path.suffix in IMG_EXTS:
            images.append(path)
            if limit is not None and len(images) >= limit:
                break
    return images

def resolve_imagenet_layout(data_root: Path):
    train_dir = data_root / 'train'
    val_dir = data_root / 'val'
    if train_dir.is_dir() and val_dir.is_dir():
        return train_dir, val_dir

    for train_dir in data_root.rglob('train'):
        val_dir = train_dir.parent / 'val'
        if train_dir.is_dir() and val_dir.is_dir():
            return train_dir, val_dir
    return None, None

def content_usage_gb():
    usage = shutil.disk_usage(Path('/content'))
    return usage.used / 1024**3, usage.free / 1024**3

def assert_local_storage_guard(stage: str):
    used_gb, free_gb = content_usage_gb()
    if used_gb > MAX_CONTENT_USED_GB or free_gb < MIN_CONTENT_FREE_GB:
        raise RuntimeError(
            f'/content 磁盘保护触发: stage={stage}, used={used_gb:.1f} GiB, free={free_gb:.1f} GiB.'
            f'\n当前限制: used <= {MAX_CONTENT_USED_GB:.1f} GiB, free >= {MIN_CONTENT_FREE_GB:.1f} GiB'
        )

def cleanup_hf_cache():
    if not DELETE_HF_CACHE_AFTER_EXPORT:
        return
    if HF_CACHE_ROOT.exists():
        print(f'Removing HF cache: {HF_CACHE_ROOT}')
        shutil.rmtree(HF_CACHE_ROOT, ignore_errors=True)
    downloads_dir = HF_HOME / 'downloads'
    if downloads_dir.exists():
        print(f'Removing HF downloads cache: {downloads_dir}')
        shutil.rmtree(downloads_dir, ignore_errors=True)
    print_disk(Path('/content'))

def ensure_repo_data_link(repo_dir: Path, data_root: Path):
    data_dir = repo_dir / 'data'
    data_dir.mkdir(parents=True, exist_ok=True)
    link_path = data_dir / 'imagenet'
    if link_path.is_symlink() or link_path.exists():
        return link_path
    os.symlink(data_root, link_path, target_is_directory=True)
    return link_path

def image_record_to_file(sample_image, dst_path: Path):
    if isinstance(sample_image, dict):
        image_bytes = sample_image.get('bytes')
        image_path = sample_image.get('path')
        if image_bytes is not None:
            dst_path.write_bytes(image_bytes)
            return
        if image_path and Path(image_path).exists():
            shutil.copy2(image_path, dst_path)
            return
    if isinstance(sample_image, Image.Image):
        sample_image.save(dst_path)
        return
    raise TypeError(f'Unsupported image payload type: {type(sample_image)!r}')

def build_output_path(split_root: Path, sample: dict, sample_idx: int) -> Path:
    label = int(sample.get('label', -1))
    class_dir = split_root / (f'class_{label:04d}' if label >= 0 else 'unlabeled')
    class_dir.mkdir(parents=True, exist_ok=True)

    image_payload = sample['image']
    stem = f'{sample_idx:08d}'
    suffix = '.jpg'
    if isinstance(image_payload, dict):
        image_path = image_payload.get('path')
        if image_path:
            path_obj = Path(str(image_path))
            if path_obj.stem:
                stem = path_obj.stem
            if path_obj.suffix:
                suffix = path_obj.suffix
    return class_dir / f'{stem}{suffix}'

def stream_hf_split(split_name: str):
    return load_dataset(HF_DATASET_ID, split=split_name, streaming=True, token=HF_TOKEN)

def export_split_to_local(split_name: str, out_dir_name: str):
    split_root = DATA_ROOT / out_dir_name
    split_root.mkdir(parents=True, exist_ok=True)

    state_path = DATA_ROOT / f'.export_state_{out_dir_name}.json'
    complete_path = DATA_ROOT / f'.export_complete_{out_dir_name}'

    if complete_path.exists():
        print(f'Skip {split_name}: already exported at {split_root}')
        return

    start_idx = 0
    if state_path.exists():
        state = json.loads(state_path.read_text(encoding='utf-8'))
        start_idx = int(state.get('processed', 0))

    assert_local_storage_guard(f'before_{split_name}')
    ds = stream_hf_split(split_name)
    iterator = itertools.islice(ds, start_idx, None) if start_idx > 0 else ds

    print(f'Export split={split_name} -> {split_root} (resume from index {start_idx})')
    processed = start_idx
    for processed, sample in enumerate(iterator, start=start_idx):
        dst_path = build_output_path(split_root, sample, processed)
        if not dst_path.exists():
            image_record_to_file(sample['image'], dst_path)

        if (processed + 1) % 5000 == 0:
            state_path.write_text(json.dumps({'processed': processed + 1}, ensure_ascii=False), encoding='utf-8')
            print(f'  {split_name}: exported {processed + 1} samples')
            print_disk(Path('/content'))
            assert_local_storage_guard(f'{split_name}_{processed + 1}')

    state_path.write_text(json.dumps({'processed': processed + 1}, ensure_ascii=False), encoding='utf-8')
    complete_path.write_text('done', encoding='utf-8')
    print(f'Finished split={split_name}, total={processed + 1}')
    print_disk(Path('/content'))

for hf_split, out_dir in HF_SPLIT_TO_DIR.items():
    export_split_to_local(hf_split, out_dir)

cleanup_hf_cache()

train_dir, val_dir = resolve_imagenet_layout(DATA_ROOT)
if train_dir is None or val_dir is None:
    raise FileNotFoundError(
        'Hugging Face ???????????? train/val ???'
        f'\nDATA_ROOT={DATA_ROOT}'
    )

repo_data_link = ensure_repo_data_link(REPO_DIR, DATA_ROOT)

print('Resolved train_dir:', train_dir)
print('Resolved val_dir:', val_dir)
print('Repo data link:', repo_data_link)
print('Train image sample count:', len(find_images(train_dir, limit=8)))
print('Val image sample count:', len(find_images(val_dir, limit=8)))


In [ ]:
from pathlib import Path
import os
import shlex
import shutil
import subprocess
import sys

def build_smoke_subset(train_src: Path, val_src: Path, smoke_root: Path, train_count: int = 64, val_count: int = 16):
    if smoke_root.exists():
        shutil.rmtree(smoke_root)

    for split, src, count in [('train', train_src, train_count), ('val', val_src, val_count)]:
        dst_dir = smoke_root / split / 'dummy'
        dst_dir.mkdir(parents=True, exist_ok=True)
        for idx, img_path in enumerate(find_images(src, limit=count)):
            dst_path = dst_dir / f'{idx:05d}{img_path.suffix.lower()}'
            os.symlink(img_path, dst_path)

    return smoke_root / 'train', smoke_root / 'val'

def find_resume_ckpt(output_root: Path, experiment: str):
    ckpt_dir = output_root / experiment / 'checkpoints'
    for name in ('last_checkpoint_norm_pix.pth', 'last_checkpoint.pth'):
        path = ckpt_dir / name
        if path.exists():
            return path
    return None

def run_pretrain(train_dir: Path, val_dir: Path, output_root: Path, experiment: str, epochs: int, batch_size: int, val_batch_size: int, grad_accum_steps: int, num_workers: int, resume: Path | None = None, visualize_every: int = 10, warmup_epochs: int | None = None):
    output_root.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        'train_ssl_freq_mae.py',
        '--dataset', 'imagenet',
        '--data_root', str(DATA_ROOT),
        '--train_dir', str(train_dir),
        '--val_dir', str(val_dir),
        '--output_dir', str(output_root),
        '--experiment', experiment,
        '--epochs', str(epochs),
        '--batch_size', str(batch_size),
        '--val_batch_size', str(val_batch_size),
        '--num_workers', str(num_workers),
        '--grad_accum_steps', str(grad_accum_steps),
        '--timm_model_name', TIMM_TINY_MODEL,
        '--timm_pretrained',
        '--visualize_every', str(visualize_every),
        '--amp',
    ]

    if warmup_epochs is not None:
        cmd.extend(['--warmup_epochs', str(warmup_epochs)])

    if resume is not None and Path(resume).exists():
        cmd.extend(['--resume', str(resume)])

    print('Command:')
    print(' '.join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, check=True, cwd=str(REPO_DIR))

smoke_train_dir, smoke_val_dir = build_smoke_subset(train_dir, val_dir, SMOKE_ROOT)
print('Smoke train dir:', smoke_train_dir)
print('Smoke val dir:', smoke_val_dir)

run_pretrain(
    train_dir=smoke_train_dir,
    val_dir=smoke_val_dir,
    output_root=RUNS_ROOT,
    experiment=SMOKE_EXPERIMENT,
    epochs=1,
    batch_size=8,
    val_batch_size=8,
    grad_accum_steps=1,
    num_workers=2,
    resume=None,
    visualize_every=1,
    warmup_epochs=1,
)

print('Smoke test finished. If this cell succeeded, you can start the full pretrain.')


## 正式训练

这一格会直接启动正式预训练。

- 训练数据使用完整 ImageNet
- checkpoint 自动保存在 `RUNS_ROOT/FULL_EXPERIMENT/`
- 如果之前已经训练过，会优先从 `last_checkpoint_norm_pix.pth` 自动续跑
- 如果显存不够，先把上面配置格里的 `TRAIN_BATCH_SIZE` 再降一档


In [ ]:
resume_ckpt = find_resume_ckpt(RUNS_ROOT, FULL_EXPERIMENT)
print('Resume checkpoint:', resume_ckpt if resume_ckpt is not None else 'none')

run_pretrain(
    train_dir=train_dir,
    val_dir=val_dir,
    output_root=RUNS_ROOT,
    experiment=FULL_EXPERIMENT,
    epochs=FULL_EPOCHS,
    batch_size=TRAIN_BATCH_SIZE,
    val_batch_size=VAL_BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    num_workers=NUM_WORKERS,
    resume=resume_ckpt,
    visualize_every=10,
    warmup_epochs=None,
)


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/imagenet_ssl_runs


训练结束后，关键输出位置：

- 最优模型：`/content/drive/MyDrive/imagenet_ssl_runs/vit_tiny_ssl_imagenet_colab/best_freq-mae-ssl.pth`
- 最新断点：`/content/drive/MyDrive/imagenet_ssl_runs/vit_tiny_ssl_imagenet_colab/checkpoints/last_checkpoint_norm_pix.pth`
- TensorBoard 日志：`/content/drive/MyDrive/imagenet_ssl_runs/vit_tiny_ssl_imagenet_colab/`

如果你后面还要在这个仓库里继续做第二阶段训练，这个 `best_freq-mae-ssl.pth` 就是后续最常用的输入。